# Démonstration : valider un jeu de données avant de l'utiliser

**Public.** Étudiants de BAC+4 Data qui connaissent Python, Pandas et la séance 2 (ingestion).

**Prérequis.** Aucun fichier à télécharger : la livraison de données est fabriquée dans le notebook, avec des défauts connus à l'avance. Les fichiers sont écrits dans un dossier temporaire. La dernière partie utilise Pandera, facultatif : lancer Jupyter avec `uv run --with pandera jupyter lab`.

**Objectifs.** Constater les limites de l'inférence de types, décrire un schéma explicite avec PyArrow, écrire et relire du Parquet, détecter une dérive de schéma, appliquer des règles de qualité qui bloquent ou signalent, puis produire un compte rendu et des tests.

## Parcours

1. Une livraison de données avec des défauts
2. Ce que la lecture par défaut ne dit pas
3. Un schéma explicite avec PyArrow
4. Parquet : un fichier qui porte son schéma
5. Contrôler le schéma d'une nouvelle livraison
6. Des règles qui bloquent ou signalent
7. Des tests de qualité automatisés
8. Aperçu de Pandera
9. De la démonstration à votre projet

In [1]:
from __future__ import annotations

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pc
import pyarrow.parquet as pq

DOSSIER = Path(tempfile.mkdtemp(prefix="demo_validation_"))

pd.__version__, pa.__version__, DOSSIER

('3.0.6',
 '26.0.0',
 PosixPath('/var/folders/yr/bzm5t_1d3hq1hzmqxjm13t840000gn/T/demo_validation_isphr4sr'))

## 1. Une livraison de données avec des défauts

Un réseau de capteurs de température envoie chaque jour un fichier de relevés. Deux tables :

- `capteurs.csv` : un capteur par ligne, avec le code postal du site et sa date de pose ;
- `releves.csv` : un relevé par ligne, avec la température, l'humidité et un statut.

La cellule suivante fabrique ces fichiers et y glisse des défauts connus : température écrite avec une virgule, valeurs `-999` et `N/A` pour une mesure absente, humidités impossibles, statuts hors liste, identifiants en double, relevés d'un capteur inconnu et une date qui n'existe pas. Il n'est pas nécessaire de la détailler en séance.

In [2]:
def fabriquer_livraison(dossier: Path, graine: int = 42, n: int = 2000) -> None:
    rng = np.random.default_rng(graine)

    capteurs = pd.DataFrame({
        "id_capteur": [f"C{i:03d}" for i in range(1, 21)],
        "code_postal": rng.choice(["01000", "06000", "13001", "31000", "69001", "75011"], 20),
        "date_pose": pd.date_range("2025-01-01", periods=20, freq="15D").strftime("%Y-%m-%d"),
    })

    instants = pd.Timestamp("2026-03-01") + pd.to_timedelta(rng.integers(0, 60 * 24 * 30, n), unit="min")
    releves = pd.DataFrame({
        "id_releve": np.arange(1, n + 1),
        "id_capteur": rng.choice(capteurs["id_capteur"], n),
        "horodatage": instants.strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": np.round(rng.normal(21, 4, n), 1).astype(object),
        "humidite": np.round(rng.normal(55, 12, n)).clip(5, 95).astype(int).astype(object),
        "statut": rng.choice(["ok", "alerte", "panne"], n, p=[0.90, 0.07, 0.03]),
    })

    # Défauts injectés à des positions tirées au hasard, mais toujours les mêmes
    p = rng.choice(n, 40, replace=False)
    releves.loc[p[:15], "temperature"] = "-999"
    releves.loc[p[15:25], "temperature"] = "N/A"
    releves.loc[p[25:30], "humidite"] = 120
    releves.loc[p[30:33], "humidite"] = -5
    releves.loc[p[33:36], "statut"] = "inconnu"
    releves.loc[p[36:38], "id_capteur"] = "C999"
    releves.loc[p[38], "horodatage"] = "2026-02-30 10:00:00"
    releves = pd.concat([releves, releves.iloc[[10, 500, 900]]], ignore_index=True)

    # Virgule décimale à la française
    releves["temperature"] = releves["temperature"].map(
        lambda v: str(v).replace(".", ",") if isinstance(v, float) else v
    )
    capteurs.to_csv(dossier / "capteurs.csv", sep=";", index=False)
    releves.to_csv(dossier / "releves.csv", sep=";", index=False)


fabriquer_livraison(DOSSIER)
print((DOSSIER / "releves.csv").read_text().splitlines()[:5])

['id_releve;id_capteur;horodatage;temperature;humidite;statut', '1;C007;2026-03-16 00:15:00;22,9;45;ok', '2;C014;2026-03-12 02:58:00;22,0;43;ok', '3;C014;2026-03-06 11:26:00;20,6;42;ok', '4;C012;2026-03-28 19:16:00;19,3;48;ok']


## 2. Ce que la lecture par défaut ne dit pas

Aucun type n'est écrit dans un CSV. Pandas en choisit un pour chaque colonne, sans rien signaler.

In [3]:
releves_brut = pd.read_csv(DOSSIER / "releves.csv", sep=";")
capteurs_brut = pd.read_csv(DOSSIER / "capteurs.csv", sep=";")

print(releves_brut.dtypes, end="\n\n")
capteurs_brut.head(3)

id_releve      int64
id_capteur       str
horodatage       str
temperature      str
humidite       int64
statut           str
dtype: object



,id_capteur,code_postal,date_pose
0,C001,1000,2025-01-01
1,C002,69001,2025-01-16
2,C003,31000,2025-01-31


Deux pièges, sans la moindre erreur :

- `temperature` est du texte : la virgule décimale empêche d'y voir un nombre ;
- `code_postal` est un entier : `01000` est devenu `1000`, le zéro de tête a disparu.

### Convertir, et compter ce qui échoue

On remplace la virgule par un point, puis on convertit. `errors="coerce"` transforme toute valeur non convertible en valeur manquante : il faut donc compter ces échecs, sinon ils disparaissent.

In [4]:
texte = releves_brut["temperature"]
temperature = pd.to_numeric(texte.str.replace(",", "."), errors="coerce")

echecs = temperature.isna() & texte.notna()
print("conversions en échec :", int(echecs.sum()))
print("valeurs manquantes (N/A) :", int(texte.isna().sum()))
print("minimum :", temperature.min())
print("moyenne :", round(temperature.mean(), 2))

conversions en échec : 0
valeurs manquantes (N/A) : 10
minimum : -999.0
moyenne : 13.42


Aucune conversion n'a échoué, et pourtant la moyenne est fausse : `-999` est un nombre valide, c'est une **valeur sentinelle** qui signifie « mesure absente ». Un contrôle de type ne suffit pas. Il faudra une règle sur les valeurs, dans la partie 6.

## 3. Un schéma explicite avec PyArrow

Un schéma décrit chaque colonne : son nom, son type et si elle peut être vide. On l'écrit une fois, à partir de la documentation de la source, et on le fait respecter à la lecture.

L'horodatage est d'abord déclaré en texte ; on verra pourquoi un peu plus bas.

In [5]:
SCHEMA = pa.schema([
    pa.field("id_releve", pa.int64(), nullable=False),
    pa.field("id_capteur", pa.string(), nullable=False),
    pa.field("horodatage", pa.string()),
    pa.field("temperature", pa.float64()),
    pa.field("humidite", pa.int16()),
    pa.field("statut", pa.string()),
])
SCHEMA

id_releve: int64 not null
id_capteur: string not null
horodatage: string
temperature: double
humidite: int16
statut: string

### Une erreur claire plutôt qu'un type deviné

Sans préciser la virgule décimale, PyArrow refuse de convertir et nomme la valeur en cause.

In [6]:
PARSE = pc.ParseOptions(delimiter=";")

try:
    pc.read_csv(
        DOSSIER / "releves.csv",
        parse_options=PARSE,
        convert_options=pc.ConvertOptions(column_types=SCHEMA),
    )
except pa.ArrowInvalid as erreur:
    print(type(erreur).__name__, ":", erreur)

ArrowInvalid : In CSV column #3: CSV conversion error to double: invalid value '22,9'


### La lecture avec les bonnes options

`decimal_point` indique la virgule. `null_values` liste les écritures d'une valeur absente, y compris la sentinelle `-999`.

In [7]:
OPTIONS = pc.ConvertOptions(
    column_types=SCHEMA,
    decimal_point=",",
    null_values=["", "N/A", "-999"],
)

table = pc.read_csv(DOSSIER / "releves.csv", parse_options=PARSE, convert_options=OPTIONS)

print(table.schema, end="\n\n")
print("lignes :", table.num_rows)
print("températures vides :", table["temperature"].null_count)

id_releve: int64
id_capteur: string
horodatage: string
temperature: double
humidite: int16
statut: string

lignes : 2003
températures vides : 25


Remarque : `nullable=False` est une **déclaration**. Le lecteur CSV ne la fait pas respecter : le champ lu reste nullable. La vérification des valeurs vides est donc une règle à écrire, ce que l'on fera dans la partie 6.

In [8]:
table.schema.field("id_releve").nullable

True

### Pourquoi l'horodatage est lu en texte

Déclarons l'horodatage en vrai type date. Une seule date impossible, le 30 février, suffit à faire échouer toute la lecture.

In [9]:
schema_date = SCHEMA.set(2, pa.field("horodatage", pa.timestamp("s")))

try:
    pc.read_csv(
        DOSSIER / "releves.csv",
        parse_options=PARSE,
        convert_options=pc.ConvertOptions(
            column_types=schema_date, decimal_point=",", null_values=["", "N/A", "-999"]
        ),
    )
except pa.ArrowInvalid as erreur:
    print(type(erreur).__name__, ":", erreur)

ArrowInvalid : In CSV column #2: CSV conversion error to timestamp[s]: invalid value '2026-02-30 10:00:00'


C'est un choix de conception. Rejeter toute la livraison pour une seule date impossible est parfois trop sévère. On lit donc l'horodatage en texte, puis on le valide par une règle, que l'on pourra classer comme avertissement et non comme erreur bloquante.

## 4. Parquet : un fichier qui porte son schéma

Écrire la table en Parquet conserve les types dans le fichier. On peut lire le schéma sans charger les données, et ne lire que les colonnes utiles.

In [10]:
chemin_parquet = DOSSIER / "releves.parquet"
pq.write_table(table, chemin_parquet)

print(pq.read_schema(chemin_parquet), end="\n\n")
print("lignes (métadonnées seules) :", pq.read_metadata(chemin_parquet).num_rows)

seules = pq.read_table(chemin_parquet, columns=["id_releve", "temperature"])
print("colonnes lues :", seules.column_names)

id_releve: int64
id_capteur: string
horodatage: string
temperature: double
humidite: int16
statut: string

lignes (métadonnées seules) : 2003
colonnes lues : ['id_releve', 'temperature']


In [11]:
pd.DataFrame({
    "format": ["CSV", "Parquet"],
    "taille_Ko": [
        (DOSSIER / "releves.csv").stat().st_size / 1024,
        chemin_parquet.stat().st_size / 1024,
    ],
}).round(1)

,format,taille_Ko
0,CSV,79.9
1,Parquet,34.1


### Pandas avec le backend Arrow

On peut rester dans Pandas tout en gardant des types Arrow. Le schéma décidé reste le même d'un outil à l'autre.

In [12]:
via_pandas = pd.read_csv(
    DOSSIER / "releves.csv", sep=";", decimal=",", dtype_backend="pyarrow"
)
print(via_pandas.dtypes, end="\n\n")

donnees = table.to_pandas()
donnees.dtypes

id_releve       int64[pyarrow]
id_capteur     string[pyarrow]
horodatage     string[pyarrow]
temperature    double[pyarrow]
humidite        int64[pyarrow]
statut         string[pyarrow]
dtype: object



id_releve        int64
id_capteur         str
horodatage         str
temperature    float64
humidite         int16
statut             str
dtype: object

## 5. Contrôler le schéma d'une nouvelle livraison

Une source évolue sans prévenir. Avant de charger un fichier, on compare son schéma à celui que l'on attend. Plutôt qu'un simple oui ou non, la fonction suivante dit **ce qui a changé**.

In [13]:
def ecarts_schema(attendu: pa.Schema, observe: pa.Schema) -> dict[str, object]:
    a = {champ.name: champ.type for champ in attendu}
    o = {champ.name: champ.type for champ in observe}
    return {
        "manquantes": sorted(a.keys() - o.keys()),
        "en_trop": sorted(o.keys() - a.keys()),
        "types": {
            nom: (str(a[nom]), str(o[nom]))
            for nom in a.keys() & o.keys()
            if a[nom] != o[nom]
        },
    }


def verifier_livraison(chemin: Path, attendu: pa.Schema) -> dict[str, object]:
    ecarts = ecarts_schema(attendu, pq.read_schema(chemin))
    if ecarts["manquantes"] or ecarts["types"]:
        raise ValueError(f"Schéma non conforme : {ecarts}")
    return ecarts

La livraison du jour est conforme. Les colonnes en trop, s'il y en avait, seraient seulement signalées.

In [14]:
verifier_livraison(chemin_parquet, SCHEMA)

{'manquantes': [], 'en_trop': [], 'types': {}}

Le lendemain, la source a changé trois choses : `temperature` est renommée `temp_c`, la colonne `statut` a disparu, `humidite` est devenue du texte, et une colonne `batterie` est apparue. Le fichier n'a pas été chargé : seul son schéma a été lu.

In [15]:
livraison = (
    table
    .rename_columns(["id_releve", "id_capteur", "horodatage", "temp_c", "humidite", "statut"])
    .drop_columns(["statut"])
)
livraison = livraison.set_column(
    livraison.schema.get_field_index("humidite"), "humidite", livraison["humidite"].cast(pa.string())
)
livraison = livraison.append_column("batterie", pa.array([90] * livraison.num_rows, pa.int8()))

chemin_derive = DOSSIER / "releves_lendemain.parquet"
pq.write_table(livraison, chemin_derive)

try:
    verifier_livraison(chemin_derive, SCHEMA)
except ValueError as erreur:
    print(erreur)

Schéma non conforme : {'manquantes': ['statut', 'temperature'], 'en_trop': ['batterie', 'temp_c'], 'types': {'humidite': ('int16', 'string')}}


## 6. Des règles qui bloquent ou signalent

Le schéma est bon, mais les valeurs ont-elles du sens ? Chaque règle renvoie un résultat structuré : son nom, son verdict, le nombre de lignes en défaut et sa gravité.

Trois verdicts possibles :

- `OK` : aucune ligne en défaut ;
- `TOLÉRÉ` : des lignes en défaut, mais sous le seuil accepté ;
- `ÉCHEC` : au-dessus du seuil.

Une règle **bloquante** en échec arrête le pipeline. Une règle de gravité **avertissement** laisse une trace sans arrêter.

In [16]:
def regle(nom: str, lignes_en_defaut: pd.Series, *, bloquante: bool, seuil: float = 0.0) -> dict[str, object]:
    nb_ko = int(lignes_en_defaut.sum())
    part = nb_ko / len(lignes_en_defaut)
    if nb_ko == 0:
        verdict = "OK"
    elif part <= seuil:
        verdict = "TOLÉRÉ"
    else:
        verdict = "ÉCHEC"
    return {
        "regle": nom,
        "verdict": verdict,
        "lignes_en_defaut": nb_ko,
        "part_%": round(100 * part, 2),
        "gravite": "bloquante" if bloquante else "avertissement",
    }

In [17]:
STATUTS = ["ok", "alerte", "panne"]


def controler(releves: pd.DataFrame, capteurs: pd.DataFrame) -> pd.DataFrame:
    horodatage = pd.to_datetime(releves["horodatage"], errors="coerce")
    date_pose = pd.to_datetime(releves["id_capteur"].map(capteurs.set_index("id_capteur")["date_pose"]))
    temperature = releves["temperature"]

    resultats = [
        # Non-nullité et unicité
        regle("id_releve renseigné", releves["id_releve"].isna(), bloquante=True),
        regle("id_releve unique", releves["id_releve"].duplicated(), bloquante=True),
        # Plages et domaines
        regle("humidité entre 0 et 100", ~releves["humidite"].between(0, 100), bloquante=False, seuil=0.01),
        regle("température renseignée", temperature.isna(), bloquante=False, seuil=0.02),
        regle("température entre -40 et 60", temperature.notna() & ~temperature.between(-40, 60), bloquante=False),
        regle("statut dans la liste", ~releves["statut"].isin(STATUTS), bloquante=False),
        # Format
        regle("horodatage valide", horodatage.isna(), bloquante=False, seuil=0.001),
        regle("code postal à 5 chiffres", ~capteurs["code_postal"].str.fullmatch(r"\d{5}"), bloquante=True),
        # Entre tables et entre colonnes
        regle("capteur connu", ~releves["id_capteur"].isin(capteurs["id_capteur"]), bloquante=True),
        regle("relevé après la pose du capteur", horodatage < date_pose, bloquante=False),
    ]
    return pd.DataFrame(resultats)

In [18]:
capteurs = pd.read_csv(DOSSIER / "capteurs.csv", sep=";", dtype=str)

rapport = controler(donnees, capteurs)
rapport

,regle,verdict,lignes_en_defaut,part_%,gravite
0,id_releve renseigné,OK,0,0.00,bloquante
1,id_releve unique,ÉCHEC,3,0.15,bloquante
2,humidité entre 0 et 100,TOLÉRÉ,8,0.40,avertissement
3,température renseignée,TOLÉRÉ,25,1.25,avertissement
4,température entre -40 et 60,OK,0,0.00,avertissement
5,statut dans la liste,ÉCHEC,3,0.15,avertissement
6,horodatage valide,TOLÉRÉ,1,0.05,avertissement
7,code postal à 5 chiffres,OK,0,0.00,bloquante
8,capteur connu,ÉCHEC,2,0.10,bloquante
9,relevé après la pose du capteur,OK,0,0.00,avertissement


Le compte rendu se lit d'un coup d'œil. Reste la décision : le pipeline continue-t-il ?

In [19]:
def decider(rapport: pd.DataFrame) -> None:
    bloquants = rapport[(rapport["gravite"] == "bloquante") & (rapport["verdict"] == "ÉCHEC")]
    if not bloquants.empty:
        raise ValueError("Pipeline arrêté, règles bloquantes en échec : " + ", ".join(bloquants["regle"]))


try:
    decider(rapport)
except ValueError as erreur:
    print(erreur)

Pipeline arrêté, règles bloquantes en échec : id_releve unique, capteur connu


Le pipeline s'arrête pour deux raisons : des doublons de clé et des relevés d'un capteur inconnu. Le traitement de ces lignes est une décision métier, pas technique. On suppose ici que le métier demande de garder le premier relevé de chaque identifiant et d'écarter les relevés sans capteur connu, en conservant leur nombre dans le compte rendu.

In [20]:
avant = len(donnees)
corrigees = donnees.drop_duplicates("id_releve", keep="first")
corrigees = corrigees[corrigees["id_capteur"].isin(capteurs["id_capteur"])]
print("lignes écartées :", avant - len(corrigees))

rapport_corrige = controler(corrigees, capteurs)
decider(rapport_corrige)
rapport_corrige

lignes écartées : 5


,regle,verdict,lignes_en_defaut,part_%,gravite
0,id_releve renseigné,OK,0,0.00,bloquante
1,id_releve unique,OK,0,0.00,bloquante
2,humidité entre 0 et 100,TOLÉRÉ,8,0.40,avertissement
3,température renseignée,TOLÉRÉ,25,1.25,avertissement
4,température entre -40 et 60,OK,0,0.00,avertissement
5,statut dans la liste,ÉCHEC,3,0.15,avertissement
6,horodatage valide,TOLÉRÉ,1,0.05,avertissement
7,code postal à 5 chiffres,OK,0,0.00,bloquante
8,capteur connu,OK,0,0.00,bloquante
9,relevé après la pose du capteur,OK,0,0.00,avertissement


Le pipeline continue. Il reste des avertissements : des statuts hors liste, des températures manquantes sous le seuil, une date impossible. Ils sont consignés dans le compte rendu, sans bloquer.

## 7. Des tests de qualité automatisés

Chaque contrôle devient une fonction qui échoue avec un message clair. Ces fonctions s'exécutent à chaque lancement du pipeline. Rangées dans un fichier `test_qualite.py`, elles se lancent aussi avec `pytest`.

In [21]:
def test_schema_conforme(table: pa.Table) -> None:
    ecarts = ecarts_schema(SCHEMA, table.schema)
    assert not ecarts["manquantes"] and not ecarts["types"], ecarts


def test_identifiants_uniques(df: pd.DataFrame) -> None:
    doublons = int(df["id_releve"].duplicated().sum())
    assert doublons == 0, f"{doublons} identifiants en double"


def test_capteurs_connus(df: pd.DataFrame) -> None:
    inconnus = int((~df["id_capteur"].isin(capteurs["id_capteur"])).sum())
    assert inconnus == 0, f"{inconnus} relevés d'un capteur inconnu"


def executer(tests: list) -> None:
    for fonction, argument in tests:
        try:
            fonction(argument)
            print("OK     ", fonction.__name__)
        except AssertionError as erreur:
            print("ÉCHEC  ", fonction.__name__, "->", erreur)

In [22]:
print("Données brutes de la livraison")
executer([
    (test_schema_conforme, table),
    (test_identifiants_uniques, donnees),
    (test_capteurs_connus, donnees),
])

print()
print("Après correction")
executer([
    (test_identifiants_uniques, corrigees),
    (test_capteurs_connus, corrigees),
])

Données brutes de la livraison
OK      test_schema_conforme
ÉCHEC   test_identifiants_uniques -> 3 identifiants en double
ÉCHEC   test_capteurs_connus -> 2 relevés d'un capteur inconnu

Après correction
OK      test_identifiants_uniques
OK      test_capteurs_connus


## 8. Aperçu de Pandera

Pandera décrit le schéma **et** les règles d'un DataFrame dans un seul objet, avec un rapport d'erreurs prêt à l'emploi. Cette partie est facultative : elle suppose le lancement avec `uv run --with pandera jupyter lab`.

In [23]:
try:
    import pandera.pandas as pda
except ImportError:
    print("Pandera n'est pas installé. Relancer Jupyter avec : uv run --with pandera jupyter lab")
else:
    schema_pandera = pda.DataFrameSchema(
        {
            "id_releve": pda.Column(int, unique=True),
            "humidite": pda.Column(int, pda.Check.in_range(0, 100)),
            "statut": pda.Column(str, pda.Check.isin(STATUTS)),
        },
        strict="filter",
    )
    try:
        schema_pandera.validate(donnees, lazy=True)
    except pda.errors.SchemaErrors as erreurs:
        print(erreurs.failure_cases[["column", "check", "failure_case"]].head(8))

Pandera n'est pas installé. Relancer Jupyter avec : uv run --with pandera jupyter lab


## 9. De la démonstration à votre projet

Ce que vous venez de voir tient en quatre questions, à poser sur votre propre projet :

1. **Ma source.** Où sont mes données, et quelle documentation les décrit ?
2. **Mon schéma.** Quel type attendu pour chaque colonne, y compris les identifiants, les codes et les montants ?
3. **Mes règles.** Quelles règles découlent de mon problème, et lesquelles sont bloquantes ?
4. **Mes contrôles.** Où les placer dans le pipeline, et que faire quand l'un d'eux échoue ?

Étape suivante : appliquer cette démarche à la mission Boutique, en classe, puis à votre projet.